# Prueba de estrés de featurización — §11.4-bis
# Camino B: piso del confound A natural

Mide si existe un **confound A natural** (sin inyección) que el oráculo actual no captura.
La correlación profundidad↔S_t proviene solo de la mecánica del juego (huella de historia compartida).

## Qué se corre

1. **Fill = `π_fill` bag-ciego estándar.** Sin `bag_en_relleno_fill`. El tablero evoluciona por mecánica natural.
2. **No-tracker sensible a profundidad** (weight `alpha_depth=0.4`, declarado). Bag-ciego: reacciona a la profundidad que ve, no a S_t.
3. **Oráculo L2 sin features de profundidad vertical.** Tiene `res_col_holes{c}` (cuenta) pero no `res_col_hole_depth{c}`. Ese es el gap a testear. Remedio si el piso se aparta: `--extend_oracle`.

## Diagnóstico pre-regresión (obligatorio)
`corr(n_JL_restantes, profundidad_tablero_en_decisión)` — global y por bin de H.  
Si ≈0: no hay confound natural que cerrar (resultado válido).  
Si ≠0: el canal existe; ver IC del piso.

## Regla de lectura del IC (escala de señal ~2–3)
- **IC angosto ≈ 0** → suficiencia demostrada sobre el caso real.
- **IC angosto lejos de 0** → remedio: añadir `res_col_hole_depth{c}` al oráculo.
- **IC ancho** (llega a magnitudes de la señal) → no concluido, falta potencia.

No interpretar β≈0 como "limpio" sin mirar el ancho del IC.

---
*Enfoque de inyección (bag_en_relleno_fill) archivado en `confound_floor_stress_t2.py`.*

In [ ]:
# ============================================================
# CONFIGURACIÓN
# ============================================================
REPO_URL = "https://github.com/yudocyudoc/tetris.git"

N_GAMES     = 300   # IC angosto requiere más partidas que el diagnóstico de inyección
MAX_PIECES  = 500
TAU         = 10.0
BVW         = 0.5
K           = 5
ALPHA_DEPTH = 0.4   # Peso de profundidad en la política del no-tracker (declarado)
N_WORKERS   = 4     # Colab Plus. Cambiar a 1 si hay deadlock.

OUT_DIR_BASE = "/content/tetris/colab/results_colab"

print(f"MODO: Camino B — confound A natural")
print(f"N_GAMES={N_GAMES}, ALPHA_DEPTH={ALPHA_DEPTH}, N_WORKERS={N_WORKERS}")

In [ ]:
# ============================================================
# CLONAR REPO E INSTALAR DEPENDENCIAS
# ============================================================
import os

if not os.path.exists("/content/tetris"):
    !git clone {REPO_URL} /content/tetris
else:
    !cd /content/tetris && git pull

!pip install -q pygame pandas pyarrow numpy scipy matplotlib tabulate statsmodels

print("Repo listo.")

In [ ]:
import os, json, subprocess, sys
import numpy as np
import pandas as pd

CONF_DIR = "/content/tetris/analysis/confound_floor"
os.makedirs(OUT_DIR_BASE, exist_ok=True)

# Script Camino B (natural, sin inyección)
NATURAL_SCRIPT = os.path.join(CONF_DIR, "confound_floor_natural_t2.py")
assert os.path.exists(NATURAL_SCRIPT), f"No se encontró {NATURAL_SCRIPT}"
print(f"Script encontrado: {NATURAL_SCRIPT}")

result = subprocess.run(["git", "rev-parse", "HEAD"],
                        capture_output=True, text=True, cwd="/content/tetris")
GIT_HASH = result.stdout.strip()
print(f"Git hash: {GIT_HASH}")

In [ ]:
# ============================================================
# PRUEBA RÁPIDA: 2 partidas — verificar que no se cuelga
# ============================================================
import time

out_dir_test = os.path.join(OUT_DIR_BASE, "out_natural_test")
os.makedirs(out_dir_test, exist_ok=True)

cmd_test = [
    sys.executable, NATURAL_SCRIPT,
    "--n_games",       "2",
    "--max_pieces",    "100",
    "--tau",           str(TAU),
    "--board_value_weight", str(BVW),
    "--k",             str(K),
    "--H_min",         "4",
    "--H_max",         "15",
    "--alpha_depth",   str(ALPHA_DEPTH),
    "--n_workers",     str(N_WORKERS),
    "--out",           out_dir_test,
]

print(f">>> Prueba rápida (2 partidas, N_WORKERS={N_WORKERS})...")
t0 = time.time()
process = subprocess.Popen(
    cmd_test, cwd=CONF_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
TIMEOUT = 90
try:
    stdout, _ = process.communicate(timeout=TIMEOUT)
    elapsed = time.time() - t0
    print(stdout)
    print(f"OK — terminó en {elapsed:.1f}s (exit code: {process.returncode})")
    if process.returncode != 0:
        print("ERROR: exit code no-cero.")
    else:
        print("Prueba superada — proceder con corrida principal.")
except subprocess.TimeoutExpired:
    process.kill()
    process.communicate()
    print(f"TIMEOUT ({TIMEOUT}s). Cambiar N_WORKERS=1 y reiniciar runtime.")

In [ ]:
# ============================================================
# CORRIDA PRINCIPAL — 300 partidas, fill natural
# ============================================================
import time

OUT_DIR_NATURAL = os.path.join(OUT_DIR_BASE, "out_natural_k5")
os.makedirs(OUT_DIR_NATURAL, exist_ok=True)

cmd = [
    sys.executable, NATURAL_SCRIPT,
    "--n_games",       str(N_GAMES),
    "--max_pieces",    str(MAX_PIECES),
    "--tau",           str(TAU),
    "--board_value_weight", str(BVW),
    "--k",             str(K),
    "--H_min",         "4",
    "--H_max",         "15",
    "--alpha_depth",   str(ALPHA_DEPTH),
    "--n_workers",     str(N_WORKERS),
    "--out",           OUT_DIR_NATURAL,
]

print(f">>> Corrida principal ({N_GAMES} partidas, alpha_depth={ALPHA_DEPTH})...")
t0 = time.time()

process = subprocess.Popen(
    cmd, cwd=CONF_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
for line in process.stdout:
    print(line, end="")
rc = process.wait()

elapsed = time.time() - t0
print(f"\nTiempo: {elapsed:.0f}s — exit code: {rc}")

with open(os.path.join(OUT_DIR_NATURAL, "git_hash.txt"), "w") as f:
    f.write(GIT_HASH + "\n")

json_path = os.path.join(OUT_DIR_NATURAL, f"natural_results_k{K}.json")
natural_results = None
if os.path.exists(json_path):
    with open(json_path) as f:
        natural_results = json.load(f)
    print(f"Resultados cargados: {list(natural_results['bin_results'].keys())}")
else:
    print(f"ADVERTENCIA: no se encontró {json_path}")

In [ ]:
# ============================================================
# DIAGNÓSTICO — correlación natural profundidad↔S_t
# ============================================================
# Antes de leer la regresión: ¿existe el confound?
# Si corr ≈ 0 en todos los bins → no hay confound A natural que cerrar.
#   Ese también es un resultado válido y citable.
# Si corr ≠ 0 en algún bin → el canal existe; mirar IC del piso.

from scipy import stats as scipy_stats

print("=" * 65)
print("DIAGNÓSTICO CONFOUND NATURAL")
print("corr(n_JL_restantes, profundidad_tablero_en_decisión)")
print("=" * 65)

csv_path = os.path.join(OUT_DIR_NATURAL, "decisions_natural_k5.csv")
if not os.path.exists(csv_path):
    print("CSV no encontrado — correr celda anterior primero.")
else:
    df_csv = pd.read_csv(csv_path)
    df_d = df_csv[df_csv["alternative_id"] == 0].copy()

    H_BIN_LABELS = ["4-6", "7-8", "9-10", "11-12", "13-15"]

    # Global
    r_g, p_g = scipy_stats.pearsonr(df_d["n_jl_now"], df_d["board_depth_at_decision"])
    print(f"\n  {'GLOBAL':<8}  n={len(df_d):>6}  corr={r_g:+.4f}  (p={p_g:.4f})")

    # Por bin
    for label in H_BIN_LABELS:
        sub = df_d[df_d["H_bin"] == label] if "H_bin" in df_d.columns else \
              df_d[df_d["H"].between(
                  int(label.split("-")[0]), int(label.split("-")[1]))]
        if len(sub) < 20:
            print(f"  {label:<8}  n={len(sub):>6}  — insuficiente")
            continue
        r, p = scipy_stats.pearsonr(sub["n_jl_now"], sub["board_depth_at_decision"])
        sig = "**" if p < 0.01 else ("*" if p < 0.05 else "  ")
        print(f"  {label:<8}  n={len(sub):>6}  corr={r:+.4f}  (p={p:.4f}) {sig}")

    print()
    print("Interpretación:")
    print("  corr ≈ 0 en todos: no hay confound A natural → IC del piso es el veredicto.")
    print("  corr ≠ 0 en algún bin: canal natural existe → leer IC del piso por bin.")

In [ ]:
# ============================================================
# TABLA: piso oracle por bin — veredicto por IC
# ============================================================
from tabulate import tabulate

SIGNAL_SCALE = 2.5  # magnitud típica de β_señal en estos bins
H_BIN_LABELS = ["4-6", "7-8", "9-10", "11-12", "13-15"]

if natural_results is None:
    print("Sin resultados — correr celda de corrida principal.")
else:
    rows = []
    for label in H_BIN_LABELS:
        bin_data = natural_results["bin_results"].get(label, {})
        ob = bin_data.get("oracle", {}).get("beta")
        ci_lo = bin_data.get("oracle", {}).get("ci_low")
        ci_hi = bin_data.get("oracle", {}).get("ci_high")
        n = bin_data.get("n_decisions", "—")

        if ob is None:
            rows.append([f"H={label}", str(n), "n/d", "n/d", "—"])
            continue

        ci_str = f"({ci_lo:+.3f}, {ci_hi:+.3f})" if ci_lo is not None else "n/d"
        ci_width = (ci_hi - ci_lo) if ci_lo is not None else float("inf")

        if ci_width > SIGNAL_SCALE:
            verdict = "IC ancho — no concluido"
        elif abs(ob) < 0.3:
            verdict = "suficiente"
        else:
            verdict = "añadir profundidad"

        rows.append([f"H={label}", str(n), f"{ob:+.3f}", ci_str, verdict])

    headers = ["Bin", "N_dec", "β oracle", "IC 95%", "Veredicto"]
    print("Piso oracle (β señal no-tracker) — Camino B natural")
    print(tabulate(rows, headers=headers, tablefmt="grid"))
    print()
    print(f"Escala de señal de referencia: β_señal ≈ {SIGNAL_SCALE}")
    print("IC angosto: ancho < escala. IC ancho: ancho ≥ escala.")

In [ ]:
# ============================================================
# TEST DEL REMEDIO — --extend_oracle (solo si IC angosto lejos de 0)
# ============================================================
# Añade res_col_hole_depth{c} al oráculo y re-corre.
# Si el piso cierra con esta feature, el gap es real y el remedio es añadir profundidad.

FORCE_REMEDY = False  # Cambiar a True para forzar el test independientemente

any_far = natural_results is not None and any(
    bin_data.get("oracle", {}).get("ci_low") is not None
    and abs(bin_data.get("oracle", {}).get("beta", 0)) >= 0.3
    and (bin_data.get("oracle", {}).get("ci_high", 0)
         - bin_data.get("oracle", {}).get("ci_low", 0)) < 2.5
    for bin_data in (natural_results["bin_results"].values() if natural_results else [])
)

if not any_far and not FORCE_REMEDY:
    print("Sin bins 'IC angosto lejos de 0' — test del remedio no necesario.")
    print("(Cambiar FORCE_REMEDY=True para correrlo de todas formas.)")
else:
    out_dir_remedy = os.path.join(OUT_DIR_BASE, "out_natural_remedy")
    os.makedirs(out_dir_remedy, exist_ok=True)

    cmd_r = [
        sys.executable, NATURAL_SCRIPT,
        "--n_games",       str(N_GAMES),
        "--max_pieces",    str(MAX_PIECES),
        "--tau",           str(TAU),
        "--board_value_weight", str(BVW),
        "--k",             str(K),
        "--H_min",         "4",
        "--H_max",         "15",
        "--alpha_depth",   str(ALPHA_DEPTH),
        "--n_workers",     str(N_WORKERS),
        "--out",           out_dir_remedy,
        "--extend_oracle",
    ]

    print(">>> Test del remedio (--extend_oracle)...")
    process = subprocess.Popen(
        cmd_r, cwd=CONF_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in process.stdout:
        print(line, end="")
    rc = process.wait()
    print(f"exit code: {rc}")

    with open(os.path.join(out_dir_remedy, "git_hash.txt"), "w") as f:
        f.write(GIT_HASH + "\n")

    json_path_r = os.path.join(out_dir_remedy, f"natural_results_k{K}.json")
    if os.path.exists(json_path_r):
        with open(json_path_r) as f:
            remedy_data = json.load(f)
        remedy_bins = remedy_data["bin_results"]

        rows_r = []
        for label in ["4-6", "7-8", "9-10", "11-12", "13-15"]:
            base = natural_results["bin_results"].get(label, {}).get("oracle", {}) if natural_results else {}
            remed = remedy_bins.get(label, {}).get("oracle", {})
            b_ci = f"({base.get('ci_low',0):+.3f}, {base.get('ci_high',0):+.3f})" if base.get("ci_low") is not None else "n/d"
            r_ci = f"({remed.get('ci_low',0):+.3f}, {remed.get('ci_high',0):+.3f})" if remed.get("ci_low") is not None else "n/d"
            rows_r.append([
                f"H={label}",
                f"{base.get('beta',0):+.3f}" if base.get("beta") is not None else "n/d",
                b_ci,
                f"{remed.get('beta',0):+.3f}" if remed.get("beta") is not None else "n/d",
                r_ci,
            ])

        print()
        print("¿Añadir res_col_hole_depth cierra el piso?")
        print(tabulate(rows_r,
                       headers=["Bin", "β base", "IC base", "β+depth", "IC+depth"],
                       tablefmt="grid"))

In [ ]:
# ============================================================
# VEREDICTO FINAL
# ============================================================
SIGNAL_SCALE = 2.5

if natural_results is None:
    print("Sin resultados.")
else:
    print("=" * 60)
    print("VEREDICTO — Camino B: confound A natural")
    print("=" * 60)

    n_sufic, n_lejos, n_ancho, n_nd = 0, 0, 0, 0
    for label, bin_data in natural_results["bin_results"].items():
        ob = bin_data.get("oracle", {}).get("beta")
        ci_lo = bin_data.get("oracle", {}).get("ci_low")
        ci_hi = bin_data.get("oracle", {}).get("ci_high")
        if ob is None or ci_lo is None:
            n_nd += 1
            continue
        ci_w = ci_hi - ci_lo
        if ci_w >= SIGNAL_SCALE:
            n_ancho += 1
        elif abs(ob) < 0.3:
            n_sufic += 1
        else:
            n_lejos += 1

    print(f"\n  Bins 'IC angosto ≈ 0' (suficiencia)      : {n_sufic}")
    print(f"  Bins 'IC angosto lejos de 0' (remedio)    : {n_lejos}")
    print(f"  Bins 'IC ancho' (no concluido)            : {n_ancho}")
    print(f"  Bins sin datos                            : {n_nd}")
    print()

    if n_ancho > 0:
        print("⚠ IC ancho en algunos bins → no concluido.")
        print("  Si la correlación natural era ≈0, el confound A no existe y")
        print("  el IC ancho es irrelevante (no hay señal que detectar).")
        print("  Si la correlación era ≠0, falta potencia — considerar n_games mayor.")
    elif n_lejos > 0:
        print("→ Piso aparte de cero con IC angosto.")
        print("  El oráculo actual no cierra el confound A natural.")
        print("  Remedio: re-correr con --extend_oracle.")
    else:
        print("→ Piso ≈ 0 con IC angosto en todos los bins.")
        print("  El oráculo actual es suficiente para el confound A natural.")
        print("  (Condicionado a que la correlación natural era medible.)")

    print()
    print(f"Git hash: {GIT_HASH}")
    print(f"N_GAMES={N_GAMES}, MAX_PIECES={MAX_PIECES}, ALPHA_DEPTH={ALPHA_DEPTH}")

In [ ]:
# ============================================================
# EMPAQUETAR RESULTADOS
# ============================================================
import shutil
from google.colab import files

zip_src = OUT_DIR_BASE
zip_dest = "/content/results_stress.zip"

shutil.make_archive("/content/results_stress", "zip", zip_src)
print(f"Zip creado: {zip_dest}")

files.download(zip_dest)
print("Descarga iniciada.")